# 🌍 Project 22 — Build a Median Composite

## 🎯 Goal

Instead of using a single Sentinel-2 image, create a **median composite** from multiple images.

The result will be a **cleaner image** with fewer clouds and less noise.

---

# 🌎 Why is this useful in Remote Sensing?

Median composites are widely used for:

- ☁ Reducing cloud contamination
- 🌫 Reducing atmospheric noise
- 🗺 Creating base maps
- 🌿 Preparing data for NDVI and classification
- 🤖 Machine learning and land cover mapping

In real research, scientists rarely rely on a single satellite image. Instead, they combine many images to produce a more reliable dataset.

---

# 🆕 New Skill: Creating Better Images

Previously (Beginner Project 18):

```text
Image Collection
       │
       ▼
   median()
       │
       ▼
Annual Composite
```

Today, we'll focus on **why** `median()` produces a better image.

```text
Many Satellite Images
        │
        ▼
Clouds
Different Lighting
Atmospheric Noise
        │
        ▼
   median()
        │
        ▼
 Cleaner Image
```

---

# 🔗 Connect to Previous Lessons

This project expands on several beginner lessons.

| Beginner Project | Skill |
|-----------------|------|
| Project 3 | Filter ImageCollection |
| Project 4 | Choose low-cloud images |
| Project 18 | Annual Composite using `median()` |
| Project 22 | Understand why `median()` creates cleaner imagery |

Today's emphasis is **understanding the concept**, not learning a completely new function.

---

# 📖 What is a Median Composite?

Imagine one pixel is observed five times.

| Image | Pixel Value |
|------|------------:|
| 1 | 180 |
| 2 | 185 |
| 3 | 188 |
| 4 | 700 ☁ |
| 5 | 190 |

One observation is affected by a cloud.

Earth Engine sorts the values:

```text
180
185
188
190
700
```

The **median** is:

```text
188
```

The cloudy value (**700**) is ignored because it is an extreme value.

Earth Engine repeats this process for **every pixel** in **every band**.

---

# 🗺 Workflow

```text
Image Collection
       │
       ▼
 Filter Date
       │
       ▼
Filter Location
       │
       ▼
   median()
       │
       ▼
Composite Image
       │
       ▼
   Display
```

---

# Step 1 — Define the Study Area

We'll continue using **Germany**.

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);
```

---

# Step 2 — Load the Sentinel-2 Collection

Instead of selecting one image, we'll collect all images from one summer season.

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-08-31');
```

Notice:

We **do not** use:

```javascript
.first()
```

because we want **many images**.

---

# ⭐ Step 3 — Create the Median Composite

This is today's key step.

```javascript
var medianComposite = collection.median();
```

### What happens here?

### Input

```text
Image Collection
├── Image 1
├── Image 2
├── Image 3
├── Image 4
└── Image 5
```

### Output

```text
One Composite Image
```

The output type changes from:

```text
ee.ImageCollection
        │
        ▼
     ee.Image
```

---

# Step 4 — Clip the Composite

```javascript
medianComposite = medianComposite.clip(countryArea);
```

This keeps only the study area.

---

# Step 5 — Visualization

```javascript
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};
```

---

# Step 6 — Display the Composite

```javascript
Map.centerObject(countryArea,5);

Map.addLayer(
  medianComposite,
  trueColor,
  'Median Composite'
);
```

You should notice that the image contains **fewer clouds** than many individual Sentinel-2 images.

---

# ⭐ Bonus — Compare with a Single Image

To appreciate the benefit of a median composite, display a single image alongside it.

```javascript
var singleImage = collection
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first();

Map.addLayer(
  singleImage,
  trueColor,
  'Single Image'
);
```

Turn the layers on and off.

Ask yourself:

- Which image contains fewer clouds?
- Which image looks more consistent?
- Which image would you use for analysis?

---

# 💻 Complete Code

```javascript
// =====================================
// 1. Study Area
// =====================================
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);

// =====================================
// 2. Load Collection
// =====================================
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-08-31');

// =====================================
// 3. Median Composite
// =====================================
var medianComposite = collection
  .median()
  .clip(countryArea);

// =====================================
// 4. Single Image (Bonus)
// =====================================
var singleImage = collection
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(countryArea);

// =====================================
// 5. Visualization
// =====================================
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};

// =====================================
// 6. Display
// =====================================
Map.centerObject(countryArea,5);

Map.addLayer(
  singleImage,
  trueColor,
  'Single Image'
);

Map.addLayer(
  medianComposite,
  trueColor,
  'Median Composite'
);
```

---

# 📝 Things to Remember

## 1. `median()`

`median()` combines an **ee.ImageCollection** into a single **ee.Image** by calculating the **median value** for every pixel.

```text
Image Collection
       │
       ▼
   median()
       │
       ▼
   One Image
```

---

## 2. Why use a Median Composite?

Compared to a single image, a median composite:

- ☁ Reduces clouds
- 🌫 Reduces atmospheric noise
- 🌍 Produces a more stable image
- 📊 Is better for analysis

---

## 3. ImageCollection vs Image

Always remember:

```text
ee.ImageCollection
        │
        ▼
     median()
        │
        ▼
      ee.Image
```

Many Earth Engine analyses require an **ee.Image**, not an **ee.ImageCollection**.

---

# ⚠ Common Beginner Mistakes

### ❌ Mistake 1: Using `.first()` instead of `.median()`

```javascript
.first()
```

This returns **one image**, which may still contain clouds or other unwanted artifacts.

---

### ❌ Mistake 2: Expecting the composite to represent one specific day

A median composite is created from observations collected over **many dates**.

It is **not** a photograph from a single moment.

---

### ❌ Mistake 3: Assuming `median()` removes every cloud

`median()` greatly reduces cloud effects, but if clouds appear in **most images** at the same location, they may still influence the result.

For the best results, **cloud masking** (such as the **QA60** method you learned in **Project 4**) is often applied before creating the composite.

---

# 💻 Your Practice (Project 22.1)

Create a **median composite** for a country or region of your choice.

### Requirements

- Create a study area.
- Load the Sentinel-2 ImageCollection.
- Filter by location and a three-month date range.
- Create a median composite using `.median()`.
- Display both:
  - A single image
  - The median composite
- Compare the two images visually.

---

# ❓ Before Writing the Code

## Question 1

Why is a median composite usually more reliable for analysis than a single satellite image?

---

## Question 2

After running:

```javascript
var medianComposite = collection.median();
```

What is the data type of `medianComposite`?

- `ee.Image`
- `ee.ImageCollection`

Explain **why**.